### Copyright Matlantis Corp. as contributors to Matlantis contrib project

##  Calculation of Polymer Free Volume
- The simulation was implemented based on [this paper](https://www.mdpi.com/2079-3197/7/2/27)
- This notebook executes nPT ensembles to equilibrate and relax the system.

In [ ]:
import json
import sys
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
from ase import units
from ase.io import read
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution, Stationary
from ase.md.npt import NPT
from ase.md import MDLogger

from pfcc_extras import show_gui
from pfcc_extras.structure.rotate import convert_atoms_to_upper
from pfcc_extras.structure.connectivity import CollisionDetector
from pfp_api_client import Estimator, ASECalculator

# Fix the random seed for reproducibility. Change the seed value to run with different random numbers.
seed = 42
np.random.seed(seed)

## Introduction
This is a sample program to execute Molecular Dynamics (MD) in the nPT ensemble using a Nose-Hoover thermostat and a Parrinello-Rahman barostat.

In nPT ensemble MD, you can simulate an atomic system where the number of particles (N), pressure (P), and temperature (T) are kept constant.
The thermostat exchanges energy with the system to maintain a constant temperature (kinetic energy). In this script, energy is controlled by the Nose-Hoover thermostat, and pressure is controlled by the Parrinello-Rahman barostat.

## Settings

## Time zone settings
This script displays the current time in the output files.
Since the Matlantis environment records time in UTC, we correct it to the local time by entering the time difference.

For Japan (JST = UTC+9), please enter timedelta(hours=9).

In [ ]:
# --- time difference ---
time_difference = timedelta(hours=9)

now = datetime.now() + time_difference
day = now.strftime("%Y%m%d")
clock = now.strftime("%H%M%S")

## Input/Output settings
Specify the input file and the output directory (folder).

It is highly recommended to prepare a separate output directory for each calculation.
This prevents file mix-ups or unintended overwrites that can occur when outputting files to the same folder.

By default, this script creates a directory inside the folder specified by out_parent named:
**_{input_file_name_without_extension}_**\_**_{memo_string}_**\_**_{date:yyyymmdd}_**\_**_{time:hhmmss}_**
and stores the results inside.

If `memo=None`, the memo part is omitted:
**_{input_file_name_without_extension}_**\_**_{date:yyyymmdd}_**\_**_{time:hhmmss}_**

If you want to define your own output folder name, please set out_dir manually.

In [ ]:
# --- input/output file settings ---'
input_file = Path('output_modeling/UItem_liq.xyz')
out_parent = Path('output_nPT')
memo = None

# Output calculation results to a folder named with the input filename and current time
if memo is None:
    out_dir = out_parent / f'{input_file.stem}_{day}_{clock}'
else:
    out_dir = out_parent / f'{input_file.stem}_{memo}_{day}_{clock}'

log_file = out_dir / f'nPT_NoseHoover_ParrinelloRahman.log'
traj_file = out_dir / f'nPT_NoseHoover_ParrinelloRahman.traj'
setting_file = out_dir / f'calc_settings.json'

## PFP settings

In [ ]:
model_version = 'v8.0.0'
calc_mode = 'R2SCAN_PLUS_D3'

## MD settings
1 ns = 1_000 ps = 1_000_000 fs  
Underscore(_) can be used to represent large number: 1000000 == 1_000_000

In [ ]:
# --- time related settings ---
time_step = 1   # fsec
num_md_steps = 1_000_000

# --- temperature and pressure related settings ---
temperature = 300  # Kelvin
pressure = 1.0  # bar

# --- thermostat and barostat related settings ---
ttime = 20.0   # tau_T thermostat time constant in fsec
pfactor = 2e6  # pressure control parameters. Notice: this is not pressure.

## Logging settings
`file_interval`: How many steps between log file outputs.  
`time_check_interval`: How many steps between elapsed time checks. Since the cost of checking the time is negligibly small, 1 is usually specified.  
`print_interval_seconds`: How many seconds should pass before printing the elapsed time to the Notebook. Printing about once every 60 seconds is useful for estimating total calculation time.

In [ ]:
file_interval = 100
time_check_interval = 1
print_interval_seconds = 60

## MISC
A collection of settings that do not affect the MD calculation results.

### Cell Reconstruction
Due to ASE specifications, when using the NPT class, the supercell must be represented as an upper triangular matrix.  
Therefore, unless there is a specific reason not to, please use `convert_to_upper_triangle=True`.  
This converts the coordinate axes of the provided input.  

### Collision Detection
This determines whether atom collisions exist in the input before starting the MD calculation.  
collision_mult: Threshold to determine a collision if an atom pair has a bond length shorter than this multiple of a standard covalent bond length.  
connection_mult: Threshold to determine if a bond length is sufficient to identify colliding parts as a molecule.

In [ ]:
# Cell reconstruction
convert_to_upper_triange = True

# Collision detection
collision_mult = 0.7
connection_mult = 1.0

## check

In [ ]:
if not input_file.exists():
    raise FileNotFoundError(f'input file: "{input_file}" is not found.')
    
print(f'Input: "{input_file}"')
print(f'Output: "{out_dir}"')

atoms = read(str(input_file))
if convert_to_upper_triange:
    atoms = convert_atoms_to_upper(atoms)
show_gui(atoms)

In [ ]:
detector = CollisionDetector(atoms, collision_mult=collision_mult, connection_mult=connection_mult)
if any(detector.is_colliding()):
    raise RuntimeError(f'Colliding atoms detected in the input. Indices:{detector.get_colliding_indices()}')

In [ ]:
# To extract and show colliding parts as molecules, uncomment the following:
# show_gui(atoms[detector.get_colliding_molecule_indices()])

# To extract and show only colliding atoms, uncomment the following:
# show_gui(atoms[detector.get_colliding_indices()])

## End of settings
# Calculation Part
## Pretreatment

In [ ]:
# === PFP ===
estimator = Estimator(model_version=model_version, calc_mode=calc_mode)
calculator = ASECalculator(estimator)
atoms.calc = calculator

# === make output directory ===
out_dir.mkdir(exist_ok=True, parents=True)

# === write settings ===
settings = {
    "force_field_params":
        {
            "method": "pfp",
            "model_version": model_version,
            "calc_mode": calc_mode,
        },
    "MD_params":
    {
        "time_step": time_step,
        "num_md_steps": num_md_steps,
        "dump_interval": file_interval,
        "ensemble": "nVT",
        "ensemble_params":
            {
                "thermostat": "Nose-Hoover",
                "temperature": temperature,
                "ttime": ttime,
                "barostat": "Parrinello-Rahman",
                "pressure": pressure,
                "pfactor": pfactor,
            }
    },
}

with open(setting_file, 'w') as f:
    json.dump(settings, f)

## Configuration of display functions

In [ ]:
from ase.utils import IOContext

class TimeLogger(IOContext):
    def __init__(self, dyn, atoms, time_difference, max_steps, logfile=sys.stdout, log_interval_seconds=60):
        self.dyn = dyn
        self.atoms = atoms
        self.logfile = logfile
        
        now = datetime.now() + time_difference
        self.start_time = now
        self.log_interval_seconds = log_interval_seconds
        self.time_difference = time_difference
        self.max_steps = max_steps
        self.header = f'{"Steps":>10}, {"Current time":>25}, {"Elapsed time":>20}, {"Estimated remain":>20}\n'
        self.header += '=' * 81 + '\n'
        self.logfile.write(self.header)
        
        steps, days, hours, minutes, seconds = 0, 0, 0, 0, 0
        timestamp = now.strftime('%Y-%m-%d %H:%M:%S')
        elapsed_time_str = f"{int(days)} days {int(hours):02}:{int(minutes):02}:{int(seconds):02}"
        log = f'{steps:>10}, {timestamp:>25}, {elapsed_time_str:>20}, {"Unknown":>20}\n'
        self.logfile.write(log)
        self.cnt = 1
        
    def __del__(self):
        self.close()
        
    def __call__(self):
        now = datetime.now() + time_difference
        if (now - self.start_time).total_seconds() <= self.log_interval_seconds * self.cnt:
            return
        
        steps = self.dyn.nsteps
        timestamp = now.strftime('%Y-%m-%d %H:%M:%S')
        
        elapsed_time = now - self.start_time
        days, remainder = divmod(elapsed_time.total_seconds(), 3600*24)
        hours, remainder = divmod(remainder, 3600)
        minutes, seconds = divmod(remainder, 60)
        elapsed_time_str = f"{int(days)} days {int(hours):02}:{int(minutes):02}:{int(seconds):02}"
        
        estimated_remain = elapsed_time * (self.max_steps - steps) / steps
        r_days, remainder = divmod(estimated_remain.total_seconds(), 3600*24)
        r_hours, remainder = divmod(remainder, 3600)
        r_minutes, r_seconds = divmod(remainder, 60)
        estimated_remain_str = f"{int(r_days)} days {int(r_hours):02}:{int(r_minutes):02}:{int(r_seconds):02}"
        
        log = f'{steps:>10}, {timestamp:>25}, {elapsed_time_str:>20}, {estimated_remain_str:>20}\n'
        
        self.before_time = now
        self.logfile.write(log)
        self.cnt +=1

## Assign initial velocities according to the Maxwell-Boltzmann distribution.

In [ ]:
# rng is the seed to fix random numbers. Added for reproducibility of the results.
MaxwellBoltzmannDistribution(atoms, temperature_K=temperature, force_temp=True, rng=np.random.RandomState(seed))
Stationary(atoms)  # Set zero total momentum to avoid drifting

## Execution of NPT

In [ ]:
dyn = NPT(
    atoms,
    time_step*units.fs,
    temperature_K = temperature,
    ttime = ttime*units.fs,
    pfactor = pfactor * units.GPa * (units.fs**2),
    externalstress=pressure*units.bar,
    loginterval=file_interval,
    trajectory=str(traj_file)
)

# set logger
# dyn.attach(print_dyn, interval=print_interval)
dyn.attach(MDLogger(dyn, atoms, str(log_file), header=True, stress=True, peratom=True, mode="w"), interval=file_interval)
dyn.attach(TimeLogger(dyn, atoms, time_difference, num_md_steps, log_interval_seconds=print_interval_seconds), interval=time_check_interval)

# run MD
dyn.run(num_md_steps)